# OpenBCI Signal Explorer: From Raw Signals to Usable Data

## Section 1 — Introduction

OpenBCI records multi-channel biosignals such as EEG and jaw-related EMG from real human sessions. Those raw recordings are valuable, but they are not immediately ready for modeling. In practice they contain environmental noise, slow drift, movement artifacts, occasional bad channels, and protocol markers that need to be interpreted carefully.

This notebook demonstrates the preprocessing pipeline using real collected data. It shows how uploaded OpenBCI sessions are parsed, inspected, filtered, and summarized before any downstream modeling is attempted. The goal is to make the signal-processing stage explicit and reproducible rather than treating it as a hidden step between data collection and machine learning.


In [1]:
import io
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, welch
from google.colab import files
from IPython.display import Markdown, display

plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "figure.dpi": 120,
        "axes.facecolor": "#fbfbfd",
        "figure.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.frameon": False,
    }
)

DEFAULT_FS_FALLBACK = 250.0
SIGNAL_NAME_RE = re.compile(r"^(ch(?:annel)?|eeg|emg|exg|adc)[ _-]*\d+$", re.IGNORECASE)
INDEX_HINTS = ("sample", "index")
TIME_HINTS = ("time", "timestamp", "ts")
MARKER_HINTS = ("marker", "event", "trigger", "label", "stim", "class")


def clean_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def robust_center(values):
    values = np.asarray(values, dtype=float)
    return values - np.nanmedian(values)


def safe_filtfilt(b, a, values):
    values = np.asarray(values, dtype=float)
    if len(values) < max(len(a), len(b)) * 3:
        return values.copy()
    finite_values = values.copy()
    if np.isnan(finite_values).any():
        fill_value = np.nanmedian(finite_values)
        if not np.isfinite(fill_value):
            fill_value = 0.0
        finite_values = np.nan_to_num(finite_values, nan=fill_value)
    return filtfilt(b, a, finite_values)


def infer_session_family(filename):
    upper_name = filename.upper()
    if "LRJ" in upper_name or "HYBRID" in upper_name:
        return "hybrid"
    if "HR" in upper_name or "JAW" in upper_name:
        return "jaw"
    if "LR" in upper_name or "LEFT" in upper_name or "RIGHT" in upper_name:
        return "left_right"
    return "unknown"


ModuleNotFoundError: No module named 'google.colab'

## Section 2 — Upload Files

The notebook is designed around uploaded files rather than hard-coded paths. Upload one or more OpenBCI recordings below. This supports multiple sessions at once, including left/right runs, jaw runs, and hybrid sessions.


In [ ]:
print("Upload one or more OpenBCI recordings (CSV or TSV exports).")
uploaded_raw = files.upload()

if not uploaded_raw:
    raise RuntimeError("No files were uploaded. Re-run this cell and select one or more files.")

UPLOADED_FILES = {name: bytes_blob for name, bytes_blob in uploaded_raw.items()}
UPLOADED_FILENAMES = sorted(UPLOADED_FILES.keys())

upload_table = pd.DataFrame(
    {
        "Index": np.arange(1, len(UPLOADED_FILENAMES) + 1),
        "Filename": UPLOADED_FILENAMES,
        "Size (KB)": [round(len(UPLOADED_FILES[name]) / 1024.0, 1) for name in UPLOADED_FILENAMES],
        "Session Guess": [infer_session_family(name) for name in UPLOADED_FILENAMES],
    }
)

display(upload_table)


def set_active_file(choice=None):
    if choice is None:
        prompt = "Enter a file index or exact filename to analyze first [1]: "
        choice = input(prompt).strip()

    if choice == "":
        choice = "1"

    if choice.isdigit():
        idx = int(choice) - 1
        if idx < 0 or idx >= len(UPLOADED_FILENAMES):
            raise IndexError(f"Selection {choice} is outside the uploaded file list.")
        selected_name = UPLOADED_FILENAMES[idx]
    else:
        if choice not in UPLOADED_FILES:
            raise KeyError(f"'{choice}' was not found in the uploaded file list.")
        selected_name = choice

    return selected_name, UPLOADED_FILES[selected_name]


ACTIVE_FILENAME, ACTIVE_BYTES = set_active_file()
print(f"Active file: {ACTIVE_FILENAME}")


## Section 3 — Automatic CSV Parsing

OpenBCI exports are not always perfectly uniform. The helper code below tries to robustly handle files with comment headers, files with or without a true CSV header row, different delimiters, varying signal column names, and marker columns that may appear at the end of the table.

The parsing pass tries to identify:

- where numeric data begins
- likely signal channels
- likely index or time columns
- a likely marker column
- an approximate sampling rate when a time-like column is available


In [ ]:
def guess_delimiter(lines, sample_count=25):
    candidates = [",", "	", ";"]
    scores = {}
    sample = lines[:sample_count]
    for delimiter in candidates:
        scores[delimiter] = sum(line.count(delimiter) for line in sample)
    best_delimiter = max(scores, key=scores.get)
    return best_delimiter if scores[best_delimiter] > 0 else ","


def looks_like_header(tokens):
    joined = " ".join(token.strip() for token in tokens)
    return any(char.isalpha() for char in joined)


def find_data_start(lines, delimiter):
    for idx, line in enumerate(lines[:80]):
        tokens = [token.strip() for token in line.split(delimiter)]
        if len(tokens) < 4:
            continue
        numeric_like = 0
        alpha_like = 0
        for token in tokens:
            if token == "":
                continue
            try:
                float(token)
                numeric_like += 1
            except ValueError:
                if any(char.isalpha() for char in token):
                    alpha_like += 1
        if numeric_like >= max(4, len(tokens) // 2):
            return idx
        if alpha_like >= 2 and idx + 1 < len(lines):
            next_tokens = [token.strip() for token in lines[idx + 1].split(delimiter)]
            next_numeric_like = 0
            for token in next_tokens:
                try:
                    float(token)
                    next_numeric_like += 1
                except ValueError:
                    continue
            if next_numeric_like >= max(4, len(next_tokens) // 2):
                return idx + 1
    return 0


def make_unique_columns(columns):
    counts = Counter()
    unique_columns = []
    for idx, name in enumerate(columns):
        candidate = str(name).strip()
        if candidate == "" or candidate.lower().startswith("unnamed"):
            candidate = f"col_{idx}"
        if candidate in counts:
            counts[candidate] += 1
            candidate = f"{candidate}_{counts[candidate]}"
        else:
            counts[candidate] = 0
        unique_columns.append(candidate)
    return unique_columns


def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def estimate_fs_from_time_series(time_series):
    numeric = clean_numeric(time_series)
    finite = numeric.dropna().to_numpy(dtype=float)
    if len(finite) < 8:
        return None, None, None

    diffs = np.diff(finite)
    diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
    if len(diffs) < 5:
        return None, None, None

    median_dt = float(np.median(diffs))
    if not np.isfinite(median_dt) or median_dt <= 0:
        return None, None, None

    fs_seconds = 1.0 / median_dt
    fs_milliseconds = 1000.0 / median_dt

    valid_index = np.flatnonzero(numeric.notna().to_numpy())
    interpolated = np.interp(np.arange(len(numeric)), valid_index, finite)

    if 20 <= fs_seconds <= 5000:
        time_axis_sec = interpolated - interpolated[0]
        return float(fs_seconds), time_axis_sec, "time column interpreted as seconds"

    if 20 <= fs_milliseconds <= 5000:
        time_axis_sec = (interpolated - interpolated[0]) / 1000.0
        return float(fs_milliseconds), time_axis_sec, "time column interpreted as milliseconds"

    return None, None, None


def detect_columns(df):
    normalized_names = {column: normalize_column_name(column) for column in df.columns}

    numeric_columns = []
    for column in df.columns:
        numeric_series = clean_numeric(df[column])
        if numeric_series.notna().mean() >= 0.80:
            numeric_columns.append(column)

    index_candidates = []
    time_candidates = []

    for column in df.columns:
        normalized = normalized_names[column]
        if any(hint in normalized for hint in INDEX_HINTS):
            index_candidates.append(column)
        if any(hint in normalized for hint in TIME_HINTS):
            time_candidates.append(column)

    for column in numeric_columns:
        numeric_series = clean_numeric(df[column]).dropna()
        if len(numeric_series) < 10:
            continue
        values = numeric_series.to_numpy(dtype=float)
        diffs = np.diff(values)
        if len(diffs) == 0:
            continue

        positive_ratio = float(np.mean(diffs > 0))
        if positive_ratio < 0.90:
            continue

        median_dt = float(np.median(np.abs(diffs)))
        if not np.isfinite(median_dt) or median_dt <= 0:
            continue

        fs_seconds = 1.0 / median_dt
        fs_milliseconds = 1000.0 / median_dt
        if 20 <= fs_seconds <= 5000 or 20 <= fs_milliseconds <= 5000:
            if column not in time_candidates:
                time_candidates.append(column)
        elif column not in index_candidates:
            index_candidates.append(column)

    marker_column = None
    marker_name_candidates = [
        column for column in df.columns if any(hint in normalized_names[column] for hint in MARKER_HINTS)
    ]
    if marker_name_candidates:
        marker_column = marker_name_candidates[0]
    else:
        low_cardinality_candidates = []
        for column in reversed(list(df.columns)):
            if column in index_candidates or column in time_candidates:
                continue
            text_series = df[column].dropna().astype(str).str.strip()
            if text_series.empty:
                continue
            unique_count = text_series.nunique()
            if unique_count <= 20:
                low_cardinality_candidates.append((column, unique_count))
        if low_cardinality_candidates:
            marker_column = low_cardinality_candidates[0][0]

    signal_candidates = [
        column for column in df.columns if SIGNAL_NAME_RE.match(normalized_names[column])
    ]
    if not signal_candidates:
        excluded = set(index_candidates + time_candidates + ([marker_column] if marker_column else []))
        ranked_numeric = []
        for column in df.columns:
            if column in excluded:
                continue
            numeric_series = clean_numeric(df[column])
            valid_fraction = float(numeric_series.notna().mean())
            unique_count = int(numeric_series.dropna().nunique())
            if valid_fraction >= 0.80 and unique_count >= 25:
                ranked_numeric.append((column, valid_fraction, unique_count))
        if not ranked_numeric:
            for column in df.columns:
                if column in excluded:
                    continue
                numeric_series = clean_numeric(df[column])
                ranked_numeric.append(
                    (column, float(numeric_series.notna().mean()), int(numeric_series.dropna().nunique()))
                )
        signal_candidates = [column for column, _, _ in ranked_numeric[:8]]

    excluded = set(index_candidates + time_candidates + ([marker_column] if marker_column else []))
    signal_candidates = [column for column in signal_candidates if column not in excluded][:8]

    return {
        "index_candidates": index_candidates,
        "time_candidates": time_candidates,
        "signal_candidates": signal_candidates,
        "marker_column": marker_column,
    }


def load_openbci_upload(file_bytes, filename):
    text = file_bytes.decode("utf-8", errors="ignore")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [line for line in text.split("\n") if line.strip() != ""]
    if not lines:
        raise ValueError(f"{filename} appears to be empty.")

    delimiter = guess_delimiter(lines)
    data_start_idx = find_data_start(lines, delimiter)
    data_tokens = [token.strip() for token in lines[data_start_idx].split(delimiter)]

    header_row_idx = None
    if data_start_idx > 0:
        previous_tokens = [token.strip() for token in lines[data_start_idx - 1].split(delimiter)]
        if len(previous_tokens) == len(data_tokens) and looks_like_header(previous_tokens):
            header_row_idx = data_start_idx - 1
    if header_row_idx is None and looks_like_header(data_tokens):
        header_row_idx = data_start_idx

    parse_start_idx = header_row_idx if header_row_idx is not None else data_start_idx
    parse_text = "\n".join(lines[parse_start_idx:])
    read_kwargs = {"sep": delimiter, "engine": "python"}
    if header_row_idx is None:
        read_kwargs["header"] = None

    df = pd.read_csv(io.StringIO(parse_text), **read_kwargs)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if df.empty:
        raise ValueError(f"{filename} did not produce a usable table after parsing.")

    if header_row_idx is None:
        df.columns = make_unique_columns([f"col_{idx}" for idx in range(df.shape[1])])
    else:
        df.columns = make_unique_columns(df.columns.tolist())

    column_info = detect_columns(df)
    fs_estimate_hz = DEFAULT_FS_FALLBACK
    fs_source = f"fallback default ({DEFAULT_FS_FALLBACK:.0f} Hz)"
    time_axis_sec = np.arange(len(df), dtype=float) / fs_estimate_hz

    if column_info["time_candidates"]:
        primary_time_column = column_info["time_candidates"][0]
        inferred_fs, inferred_time_axis_sec, inferred_source = estimate_fs_from_time_series(df[primary_time_column])
        if inferred_fs is not None:
            fs_estimate_hz = inferred_fs
            time_axis_sec = inferred_time_axis_sec
            fs_source = inferred_source

    duration_sec = float(time_axis_sec[-1]) if len(time_axis_sec) else 0.0

    marker_preview = []
    if column_info["marker_column"] is not None:
        marker_series = df[column_info["marker_column"]].dropna().astype(str).str.strip()
        marker_series = marker_series[marker_series != ""]
        marker_preview = marker_series.unique().tolist()[:12]

    return {
        "filename": filename,
        "df": df,
        "delimiter": delimiter,
        "header_detected": header_row_idx is not None,
        "family_guess": infer_session_family(filename),
        "signal_columns": column_info["signal_candidates"],
        "marker_column": column_info["marker_column"],
        "time_column": column_info["time_candidates"][0] if column_info["time_candidates"] else None,
        "fs_estimate_hz": float(fs_estimate_hz),
        "fs_source": fs_source,
        "time_axis_sec": np.asarray(time_axis_sec, dtype=float),
        "duration_sec": duration_sec,
        "marker_preview": marker_preview,
        "column_info": column_info,
    }


def print_session_summary(session):
    summary_rows = [
        ("Rows", len(session["df"])),
        ("Columns", len(session["df"].columns)),
        ("Estimated sampling rate (Hz)", round(session["fs_estimate_hz"], 2)),
        ("Sampling rate source", session["fs_source"]),
        ("Candidate signal channels", ", ".join(session["signal_columns"]) or "None detected"),
        ("Candidate marker column", session["marker_column"] or "None detected"),
        ("Session family guess", session["family_guess"]),
        ("Duration (s)", round(session["duration_sec"], 2)),
    ]
    summary_df = pd.DataFrame(summary_rows, columns=["Field", "Value"])
    display(summary_df)
    if session["marker_preview"]:
        print("Marker preview:", ", ".join(map(str, session["marker_preview"])))
    else:
        print("Marker preview: none detected")
    display(session["df"].head())


SESSION_RECORDS = {}
FAILED_UPLOADS = {}

for filename in UPLOADED_FILENAMES:
    try:
        SESSION_RECORDS[filename] = load_openbci_upload(UPLOADED_FILES[filename], filename)
    except Exception as exc:
        FAILED_UPLOADS[filename] = str(exc)

if not SESSION_RECORDS:
    raise RuntimeError("None of the uploaded files could be parsed into usable tables.")

if ACTIVE_FILENAME not in SESSION_RECORDS:
    replacement_name = next(iter(SESSION_RECORDS))
    print(f"Selected file '{ACTIVE_FILENAME}' failed to parse. Falling back to '{replacement_name}'.")
    ACTIVE_FILENAME = replacement_name

ACTIVE_SESSION = SESSION_RECORDS[ACTIVE_FILENAME]

if FAILED_UPLOADS:
    print("Some files could not be parsed automatically:")
    display(pd.DataFrame([{"Filename": name, "Error": error} for name, error in FAILED_UPLOADS.items()]))

print_session_summary(ACTIVE_SESSION)



## Section 4 — Raw Signal Visualization

The plots below show the session before any filtering. Two views are useful early in the pipeline:

- a short zoomed segment to inspect signal shape and local artifacts
- a full-session overview to inspect drift, dropout, and gross instability across time


In [ ]:
def session_signal_frame(session, use_filtered=None):
    source_df = session["df"] if use_filtered is None else use_filtered
    return source_df.loc[:, session["signal_columns"]].apply(clean_numeric)


def choose_plot_channels(session, max_channels=8):
    channels = session["signal_columns"]
    return channels[:max_channels]


def plot_stacked_traces(session, channels=None, start_sec=0.0, duration_sec=None, title="Stacked Signal View"):
    channels = channels or choose_plot_channels(session)
    if not channels:
        raise ValueError("No candidate signal channels were detected for plotting.")

    signal_df = session_signal_frame(session)
    time_axis = session["time_axis_sec"]
    mask = time_axis >= float(start_sec)
    if duration_sec is not None:
        mask &= time_axis <= float(start_sec + duration_sec)

    x = time_axis[mask]
    if len(x) == 0:
        raise ValueError("No samples fall inside the requested time window.")

    fig, ax = plt.subplots(figsize=(16, 0.9 * len(channels) + 2.5))
    offsets = np.arange(len(channels))[::-1]

    for idx, channel in enumerate(channels):
        y = clean_numeric(signal_df[channel]).to_numpy(dtype=float)[mask]
        y = robust_center(y)
        spread = np.nanpercentile(np.abs(y), 95)
        scale = spread if np.isfinite(spread) and spread > 0 else 1.0
        y_scaled = y / scale
        ax.plot(x, y_scaled + offsets[idx], linewidth=0.9, label=channel)

    ax.set_yticks(offsets)
    ax.set_yticklabels(channels)
    ax.set_xlabel("Time (s)")
    ax.set_title(title)
    ax.set_ylabel("Channel")
    return fig, ax


plot_channels = choose_plot_channels(ACTIVE_SESSION)
plot_stacked_traces(
    ACTIVE_SESSION,
    channels=plot_channels,
    start_sec=0.0,
    duration_sec=min(10.0, ACTIVE_SESSION["duration_sec"] if ACTIVE_SESSION["duration_sec"] > 0 else 10.0),
    title=f"{ACTIVE_FILENAME} — First 10 Seconds",
)
plt.show()

plot_stacked_traces(
    ACTIVE_SESSION,
    channels=plot_channels,
    start_sec=0.0,
    duration_sec=None,
    title=f"{ACTIVE_FILENAME} — Full Session Overview",
)
plt.show()


## Section 5 — Channel Quality Check

Before filtering or modeling, each channel should be screened for obvious failure modes. The table below checks for:

- flatlined or effectively constant channels
- low variance channels
- channels with unusually high point-to-point noise
- clipping or railing behavior


In [ ]:
def evaluate_channel_quality(session):
    signal_df = session_signal_frame(session)
    rows = []

    preliminary = []
    for channel in session["signal_columns"]:
        values = clean_numeric(signal_df[channel]).dropna().to_numpy(dtype=float)
        if len(values) == 0:
            rows.append(
                {
                    "Channel": channel,
                    "Variance": np.nan,
                    "Range": np.nan,
                    "Std": np.nan,
                    "Diff Std": np.nan,
                    "Dominant Value Fraction": np.nan,
                    "Rail Fraction": np.nan,
                    "Status": "FLATLINE",
                    "Notes": "No numeric samples",
                }
            )
            continue

        variance = float(np.var(values))
        value_range = float(np.ptp(values))
        std = float(np.std(values))
        diff_std = float(np.std(np.diff(values))) if len(values) > 1 else 0.0
        counts = pd.Series(np.round(values, 6)).value_counts(normalize=True)
        dominant_fraction = float(counts.iloc[0]) if not counts.empty else 1.0
        rail_eps = max(1e-6, value_range * 0.01)
        rail_fraction = float(np.mean((values <= np.min(values) + rail_eps) | (values >= np.max(values) - rail_eps)))

        preliminary.append((channel, variance, value_range, std, diff_std, dominant_fraction, rail_fraction))

    if preliminary:
        median_variance = float(np.median([row[1] for row in preliminary if np.isfinite(row[1])]))
        median_range = float(np.median([row[2] for row in preliminary if np.isfinite(row[2])]))
        median_diff_std = float(np.median([row[4] for row in preliminary if np.isfinite(row[4])]))
    else:
        median_variance = median_range = median_diff_std = 1.0

    for channel, variance, value_range, std, diff_std, dominant_fraction, rail_fraction in preliminary:
        status = "GOOD"
        notes = []

        if value_range < 1e-6 or variance < 1e-8 or dominant_fraction > 0.995:
            status = "FLATLINE"
            notes.append("signal is nearly constant")
        elif variance < max(1e-8, 0.05 * median_variance) or value_range < max(1e-5, 0.15 * median_range):
            status = "LOW VARIANCE"
            notes.append("variance is small relative to other channels")
        elif rail_fraction > 0.25 or dominant_fraction > 0.35:
            status = "CLIPPING"
            notes.append("large fraction of samples sit near extrema")
        elif diff_std > max(1e-6, 3.5 * median_diff_std):
            status = "NOISY"
            notes.append("point-to-point variation is unusually high")

        rows.append(
            {
                "Channel": channel,
                "Variance": variance,
                "Range": value_range,
                "Std": std,
                "Diff Std": diff_std,
                "Dominant Value Fraction": dominant_fraction,
                "Rail Fraction": rail_fraction,
                "Status": status,
                "Notes": "; ".join(notes) if notes else "usable",
            }
        )

    quality_df = pd.DataFrame(rows).sort_values("Channel").reset_index(drop=True)
    return quality_df


CHANNEL_QUALITY_DF = evaluate_channel_quality(ACTIVE_SESSION)
display(CHANNEL_QUALITY_DF)

GOOD_CHANNELS = CHANNEL_QUALITY_DF.loc[CHANNEL_QUALITY_DF["Status"] == "GOOD", "Channel"].tolist()
if not GOOD_CHANNELS:
    GOOD_CHANNELS = CHANNEL_QUALITY_DF.loc[
        ~CHANNEL_QUALITY_DF["Status"].isin(["FLATLINE", "CLIPPING"]), "Channel"
    ].tolist()

print("Good channels selected for downstream inspection:", GOOD_CHANNELS if GOOD_CHANNELS else "None")


## Section 6 — Filtering Pipeline

The filtering stage below implements three reusable signal-processing steps:

1. a `60 Hz` notch filter to reduce line noise
2. a `1–40 Hz` bandpass to keep broad EEG-relevant content while removing drift and high-frequency noise
3. an optional `8–30 Hz` mu/beta band for motor-intent inspection

The comparison plot shows why filtering matters before modeling.


In [ ]:
def notch_filter(values, fs_hz, notch_hz=60.0, quality_factor=30.0):
    nyquist = 0.5 * fs_hz
    if notch_hz >= nyquist:
        return np.asarray(values, dtype=float)
    b, a = iirnotch(notch_hz, quality_factor, fs_hz)
    return safe_filtfilt(b, a, values)


def bandpass_filter(values, fs_hz, low_hz=1.0, high_hz=40.0, order=4):
    nyquist = 0.5 * fs_hz
    high_hz = min(high_hz, nyquist * 0.95)
    if high_hz <= low_hz:
        return np.asarray(values, dtype=float)
    b, a = butter(order, [low_hz / nyquist, high_hz / nyquist], btype="band")
    return safe_filtfilt(b, a, values)


def filtered_signal_table(session, channels):
    fs_hz = float(session["fs_estimate_hz"])
    signal_df = session_signal_frame(session)
    filtered_1_40 = pd.DataFrame(index=signal_df.index)
    filtered_8_30 = pd.DataFrame(index=signal_df.index)

    for channel in channels:
        raw_values = clean_numeric(signal_df[channel]).to_numpy(dtype=float)
        notched = notch_filter(raw_values, fs_hz)
        filtered_1_40[channel] = bandpass_filter(notched, fs_hz, low_hz=1.0, high_hz=40.0)
        filtered_8_30[channel] = bandpass_filter(notched, fs_hz, low_hz=8.0, high_hz=30.0)

    return filtered_1_40, filtered_8_30


FILTER_CHANNELS = GOOD_CHANNELS[: min(4, len(GOOD_CHANNELS))] if GOOD_CHANNELS else choose_plot_channels(ACTIVE_SESSION, max_channels=4)
FILTERED_1_40, FILTERED_8_30 = filtered_signal_table(ACTIVE_SESSION, FILTER_CHANNELS)

example_channel = FILTER_CHANNELS[0]
raw_values = session_signal_frame(ACTIVE_SESSION)[example_channel].apply(float).to_numpy()
time_axis = ACTIVE_SESSION["time_axis_sec"]
mask = time_axis <= (min(10.0, ACTIVE_SESSION["duration_sec"]) if ACTIVE_SESSION["duration_sec"] > 0 else 10.0)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
axes[0].plot(time_axis[mask], robust_center(raw_values[mask]), color="#334155", linewidth=1.0)
axes[0].set_title(f"{example_channel} — Raw")
axes[0].set_ylabel("Amplitude")

axes[1].plot(time_axis[mask], robust_center(FILTERED_1_40[example_channel].to_numpy()[mask]), color="#0f766e", linewidth=1.0)
axes[1].set_title(f"{example_channel} — 60 Hz Notch + 1–40 Hz Bandpass")
axes[1].set_ylabel("Amplitude")

axes[2].plot(time_axis[mask], robust_center(FILTERED_8_30[example_channel].to_numpy()[mask]), color="#7c3aed", linewidth=1.0)
axes[2].set_title(f"{example_channel} — 60 Hz Notch + 8–30 Hz Mu/Beta Band")
axes[2].set_ylabel("Amplitude")
axes[2].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


## Section 7 — Marker / Event Overlay

If a marker column exists, the code below detects event changes and overlays them on the selected signal. This is useful for checking whether protocol timing looks plausible and whether left/right/jaw prompts align with meaningful signal changes.

If no marker column is detected, this section will skip gracefully.


In [ ]:
def classify_marker_value(value):
    text = str(value).strip().upper()
    if text in {"", "0", "0.0", "NAN"}:
        return None
    if "LEFT" in text or text in {"1", "1.0", "L"}:
        return "LEFT"
    if "RIGHT" in text or text in {"2", "2.0", "R"}:
        return "RIGHT"
    if "JAW" in text or "CLENCH" in text or text in {"3", "3.0", "J"}:
        return "JAW"
    return "TRANSITION"


def extract_marker_events(session):
    marker_column = session["marker_column"]
    if marker_column is None:
        return pd.DataFrame()

    marker_series = session["df"][marker_column]
    numeric_series = clean_numeric(marker_series)
    time_axis = session["time_axis_sec"]
    events = []

    if numeric_series.notna().mean() > 0.90:
        marker_values = numeric_series.fillna(0).to_numpy(dtype=float)
        change_idx = np.where(np.diff(marker_values, prepend=marker_values[0]) != 0)[0]
        event_idx = [idx for idx in change_idx if marker_values[idx] != 0]
        for idx in event_idx:
            raw_value = marker_values[idx]
            label = classify_marker_value(raw_value)
            if label is None:
                continue
            events.append(
                {
                    "sample_index": int(idx),
                    "time_sec": float(time_axis[idx]),
                    "marker_value": raw_value,
                    "event_type": label,
                }
            )
    else:
        marker_values = marker_series.fillna("").astype(str).to_numpy()
        prior = np.roll(marker_values, 1)
        prior[0] = marker_values[0]
        change_idx = np.where(prior != marker_values)[0]
        for idx in change_idx:
            raw_value = marker_values[idx].strip()
            label = classify_marker_value(raw_value)
            if label is None:
                continue
            events.append(
                {
                    "sample_index": int(idx),
                    "time_sec": float(time_axis[idx]),
                    "marker_value": raw_value,
                    "event_type": label,
                }
            )

    return pd.DataFrame(events)


MARKER_EVENTS_DF = extract_marker_events(ACTIVE_SESSION)
if MARKER_EVENTS_DF.empty:
    display(Markdown("No usable marker column was detected, so event overlay was skipped."))
else:
    display(Markdown("### Unique Marker Values"))
    display(pd.DataFrame({"Unique Marker Values": sorted(MARKER_EVENTS_DF["marker_value"].astype(str).unique())}))

    plot_channel = FILTER_CHANNELS[0] if FILTER_CHANNELS else choose_plot_channels(ACTIVE_SESSION, 1)[0]
    signal_for_overlay = FILTERED_1_40[plot_channel].to_numpy() if plot_channel in FILTERED_1_40 else session_signal_frame(ACTIVE_SESSION)[plot_channel].to_numpy()
    overlay_mask = ACTIVE_SESSION["time_axis_sec"] <= (min(30.0, ACTIVE_SESSION["duration_sec"]) if ACTIVE_SESSION["duration_sec"] > 0 else 30.0)

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(
        ACTIVE_SESSION["time_axis_sec"][overlay_mask],
        robust_center(signal_for_overlay[overlay_mask]),
        color="#1f2937",
        linewidth=1.0,
        label=plot_channel,
    )

    color_map = {"LEFT": "#2563eb", "RIGHT": "#dc2626", "JAW": "#059669", "TRANSITION": "#a16207"}
    used_labels = set()
    for _, row in MARKER_EVENTS_DF.iterrows():
        if row["time_sec"] > ACTIVE_SESSION["time_axis_sec"][overlay_mask][-1]:
            continue
        event_type = row["event_type"]
        color = color_map.get(event_type, "#475569")
        label = event_type if event_type not in used_labels else None
        ax.axvline(row["time_sec"], color=color, linestyle="--", alpha=0.8, linewidth=1.1, label=label)
        used_labels.add(event_type)

    ax.set_title(f"{plot_channel} with Marker / Event Overlay")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Filtered Amplitude")
    ax.legend(loc="upper right")
    plt.show()


## Section 8 — Frequency View

Welch power spectral density estimates help show whether a session has plausible low-frequency structure, usable beta-band activity, and obvious line-noise contamination. The plot below highlights the alpha and beta ranges and marks the line-noise region.


In [ ]:
def plot_psd(session, channels, max_freq=80.0):
    fs_hz = float(session["fs_estimate_hz"])
    signal_df = session_signal_frame(session)

    fig, ax = plt.subplots(figsize=(14, 6))
    for channel in channels:
        values = clean_numeric(signal_df[channel]).dropna().to_numpy(dtype=float)
        if len(values) < 8:
            continue
        nperseg = min(int(fs_hz * 2), len(values))
        freqs, power = welch(values, fs=fs_hz, nperseg=nperseg)
        keep = freqs <= max_freq
        ax.semilogy(freqs[keep], power[keep], linewidth=1.3, label=channel)

    ax.axvspan(8, 12, color="#dbeafe", alpha=0.6, label="Alpha (8–12 Hz)")
    ax.axvspan(13, 30, color="#ede9fe", alpha=0.4, label="Beta (13–30 Hz)")
    ax.axvline(60, color="#dc2626", linestyle="--", linewidth=1.0, label="60 Hz")
    ax.set_title("Welch Power Spectral Density")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power Spectral Density")
    ax.legend(loc="upper right", ncol=2)
    plt.show()


PSD_CHANNELS = GOOD_CHANNELS[: min(4, len(GOOD_CHANNELS))] if GOOD_CHANNELS else choose_plot_channels(ACTIVE_SESSION, 4)
plot_psd(ACTIVE_SESSION, PSD_CHANNELS)


## Section 9 — Multi-File Comparison

If multiple sessions were uploaded, the table below compares them at a high level. This is useful for deciding which recordings look strongest before deeper modeling work begins.


In [ ]:
def ensure_session_quality(session):
    if "channel_quality" not in session:
        session["channel_quality"] = evaluate_channel_quality(session)
    return session["channel_quality"]


def ensure_marker_events(session):
    if "marker_events" not in session:
        session["marker_events"] = extract_marker_events(session)
    return session["marker_events"]


def summarize_uploaded_sessions(session_records):
    rows = []
    for filename, session in session_records.items():
        quality_df = ensure_session_quality(session)
        marker_events = ensure_marker_events(session)
        good_channel_count = int((quality_df["Status"] == "GOOD").sum())
        rows.append(
            {
                "Filename": filename,
                "Session Guess": session["family_guess"],
                "Duration (s)": round(session["duration_sec"], 2),
                "Estimated Fs (Hz)": round(session["fs_estimate_hz"], 2),
                "Good Channels": f"{good_channel_count}/{max(1, len(session['signal_columns']))}",
                "Marker Count": int(len(marker_events)),
                "Marker Column": session["marker_column"] or "None",
            }
        )
    return pd.DataFrame(rows).sort_values(["Session Guess", "Filename"]).reset_index(drop=True)


MULTI_FILE_SUMMARY_DF = summarize_uploaded_sessions(SESSION_RECORDS)
display(MULTI_FILE_SUMMARY_DF)


## Section 10 — Final Interpretation

The last cell below produces a concise professor-facing interpretation of the uploaded sessions. It summarizes which files appear strongest, whether the marker structure is usable, and which sessions appear more appropriate for jaw, EEG, or hybrid modeling.


In [ ]:
def suitability_label(session):
    family = session["family_guess"]
    marker_events = ensure_marker_events(session)
    marker_types = set(marker_events["event_type"].tolist()) if not marker_events.empty else set()
    good_channels = int((ensure_session_quality(session)["Status"] == "GOOD").sum())

    suitable_for_eeg = good_channels >= 4 and (family in {"left_right", "hybrid"} or {"LEFT", "RIGHT"} & marker_types)
    suitable_for_jaw = family in {"jaw", "hybrid"} or "JAW" in marker_types
    suitable_for_hybrid = suitable_for_eeg and suitable_for_jaw

    labels = []
    if suitable_for_eeg:
        labels.append("EEG")
    if suitable_for_jaw:
        labels.append("JAW")
    if suitable_for_hybrid:
        labels.append("HYBRID")
    return ", ".join(labels) if labels else "Needs manual review"


def build_interpretation(session_records):
    ranked = []
    weak_channel_notes = []
    marker_notes = []
    suitability_rows = []

    for filename, session in session_records.items():
        quality_df = ensure_session_quality(session)
        marker_events = ensure_marker_events(session)
        good_channels = int((quality_df["Status"] == "GOOD").sum())
        usable_channels = int((~quality_df["Status"].isin(["FLATLINE", "CLIPPING"])).sum())
        ranked.append((filename, good_channels, usable_channels, len(marker_events), session["duration_sec"]))

        weak_channels = quality_df.loc[quality_df["Status"] != "GOOD", ["Channel", "Status"]]
        if not weak_channels.empty:
            weak_channel_notes.append(
                f"- **{filename}** weak channels: "
                + ", ".join(f"{row.Channel} ({row.Status})" for row in weak_channels.itertuples())
            )

        if session["marker_column"] is None:
            marker_notes.append(f"- **{filename}**: no reliable marker column was detected.")
        else:
            marker_notes.append(
                f"- **{filename}**: marker column `{session['marker_column']}` produced {len(marker_events)} detected events."
            )

        suitability_rows.append(
            {
                "Filename": filename,
                "Recommended Use": suitability_label(session),
            }
        )

    ranked.sort(key=lambda row: (row[1], row[2], row[3], row[4]), reverse=True)
    best_sessions = ", ".join(row[0] for row in ranked[: min(3, len(ranked))])

    interpretation = f"""
### Final Interpretation

**Highest-quality uploaded sessions:** {best_sessions}

**Marker usability**
{chr(10).join(marker_notes) if marker_notes else '- No marker observations available.'}

**Weak-channel review**
{chr(10).join(weak_channel_notes) if weak_channel_notes else '- No clearly weak channels were detected by the current quality screen.'}

**Recommended modeling use**
"""
    display(Markdown(interpretation))
    display(pd.DataFrame(suitability_rows))


build_interpretation(SESSION_RECORDS)
